# Librerias

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import re
import causalidad as cs

Initializing causalidad package...
causalidad package initialized successfully.


# Lectura de datos

In [3]:
df_icare = pd.read_csv(r"C:\Users\afpue\OneDrive\Documentos\GitHub\causalidad\Codigo\df_icare_260226.csv")

# Funciones

In [4]:

# 1. El PS se estima y queda guardado como columna en df
cols = ['actfreq_sq001', 'recgov_sq001']
df_icare[cols] = df_icare[cols].apply(lambda x: (x == 1).astype(int))

tratamiento = 'recgov_sq001'
resultado   = 'actfreq_sq001'
covariables = ['sex', 'age', 'edu']

df = cs.propensity_score(df_icare, tratamiento='recgov_sq001', covariables=['sex', 'age', 'edu'])

# 2. balance() lo lee directamente de ahí
df_trim  = cs.balance(df, 'recgov_sq001', metodo='trimming')
df_trunc = cs.balance(df, 'recgov_sq001', metodo='truncating')
df_match = cs.balance(df, 'recgov_sq001', metodo='matching')
df_sub   = cs.balance(df, 'recgov_sq001', metodo='subclassif')

propensity_score() — n=1663 (T=1645, C=18)
  PS: min=0.8749  p25=0.9828  media=0.9892  p75=0.9968  max=0.9999
  Aviso: 1662 unidades (99.9%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='trimming' | n=1663 (T=1645, C=18)
──────────────────────────────────────────────────
Trimming (kappa=0.05) — 1662 unidades eliminadas (99.9%)  -> n final=1  (tratados=1, controles=0)
  Rango PS conservado: [0.8749, 0.8749]
──────────────────────────────────────────────────

balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='truncating' | n=1663 (T=1645, C=18)
──────────────────────────────────────────────────
Truncating (kappa=0.05) — 1662 PS truncados (99.9%)  -> n=1663 (sin eliminar filas)
  PS original:  [0.8749, 0.9999]  ->  PS truncado: [0.8749, 0.9500]
──────────────────────────────────────────────────

bala

In [5]:
df     = cs.propensity_score(df, 'recgov_sq001', ['sex', 'age', 'edu'])
df_bal = cs.balance(df, 'recgov_sq001', metodo='matching')
res    = cs.calcular_ate(df_bal, 'actfreq_sq001', 'recgov_sq001', ['sex', 'age', 'edu'])

Aviso: la columna 'propensity_score' ya existe y será sobreescrita.
propensity_score() — n=1663 (T=1645, C=18)
  PS: min=0.8749  p25=0.9828  media=0.9892  p75=0.9968  max=0.9999
  Aviso: 1662 unidades (99.9%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='matching' | n=1663 (T=1645, C=18)
──────────────────────────────────────────────────
Matching 1:1 — 18 pares  (tratados=18, controles=18)
──────────────────────────────────────────────────



c:\Users\afpue\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [6]:
res

{'n_tratados': 18,
 'n_controles': 18,
 'naive': 0.4444,
 'regresion': 0.4253,
 'regresion_ic95': (0.1302, 0.7204),
 'regresion_pvalor': 0.0062,
 'g_formula': 0.4117,
 'ht': -25.4593,
 'hajek': 0.3957,
 'msm': 0.3957,
 'msm_ic95': (-1.1868, 1.9783),
 'msm_pvalor': 0.6146,
 'dr': -1.1347}

In [7]:
import pandas as pd

resultados_finales = []
numero_de_pares = 30 # Ajusta este número al total de pares que tengas

for i in range(1, numero_de_pares + 1):
    sufijo = f"{i:03d}" 
    col_recgov = f'recgov_sq{sufijo}'
    col_actfreq = f'actfreq_sq{sufijo}'
    
    # 1. Verificación de existencia
    if col_recgov not in df.columns or col_actfreq not in df.columns:
        continue
        
    print(f"Procesando par {sufijo}...")

    try:
        # 2. Copiamos para no alterar el DataFrame original
        df_temp = df.copy()

        # 3. CONVERSIÓN CRÍTICA: Transformar a binario (1 si es 1, 0 en cualquier otro caso)
        # Esto asegura que el modelo entienda quién recibió el "tratamiento"
        cols_par = [col_recgov, col_actfreq]
        df_temp[cols_par] = df_temp[cols_par].apply(lambda x: (x == 1).astype(int))

        # 4. Propensity Score
        df_temp = cs.propensity_score(df_temp, col_recgov, ['sex', 'age', 'edu'])
        
        # 5. Balanceo
        df_bal = cs.balance(df_temp, col_recgov, metodo='matching')
        
        # 6. Calcular ATE
        res = cs.calcular_ate(df_bal, col_actfreq, col_recgov, ['sex', 'age', 'edu'])
        
        # 7. Guardar resultados
        res_df = pd.DataFrame([res])
        res_df['pregunta_sq'] = sufijo
        resultados_finales.append(res_df)
        
    except Exception as e:
        print(f"Error en par {sufijo}: {e}")

# --- RESULTADO FINAL ---
if resultados_finales:
    df_resultados = pd.concat(resultados_finales, ignore_index=True)
    print("\n¡Listo! Tabla de resultados generada:")
    display(df_resultados)
else:
    print("No se pudo procesar ningún par. Revisa si los nombres de las columnas son correctos.")

Procesando par 001...
Aviso: la columna 'propensity_score' ya existe y será sobreescrita.
propensity_score() — n=1663 (T=1645, C=18)
  PS: min=0.8749  p25=0.9828  media=0.9892  p75=0.9968  max=0.9999
  Aviso: 1662 unidades (99.9%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='matching' | n=1663 (T=1645, C=18)
──────────────────────────────────────────────────
Matching 1:1 — 18 pares  (tratados=18, controles=18)
──────────────────────────────────────────────────



c:\Users\afpue\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Procesando par 002...
Aviso: la columna 'propensity_score' ya existe y será sobreescrita.
propensity_score() — n=1663 (T=1542, C=121)
  PS: min=0.8944  p25=0.9094  media=0.9272  p75=0.9424  max=0.9625
  Aviso: 381 unidades (22.9%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='matching' | n=1663 (T=1542, C=121)
──────────────────────────────────────────────────
Matching 1:1 — 121 pares  (tratados=121, controles=121)
──────────────────────────────────────────────────

Procesando par 003...
Aviso: la columna 'propensity_score' ya existe y será sobreescrita.
propensity_score() — n=1663 (T=1541, C=122)
  PS: min=0.8187  p25=0.9105  media=0.9266  p75=0.9470  max=0.9675
  Aviso: 382 unidades (23.0%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | me

c:\Users\afpue\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


propensity_score() — n=1663 (T=237, C=1426)
  PS: min=0.0743  p25=0.1227  media=0.1425  p75=0.1525  max=0.3059
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='matching' | n=1663 (T=237, C=1426)
──────────────────────────────────────────────────
Matching 1:1 — 237 pares  (tratados=237, controles=237)
──────────────────────────────────────────────────

Procesando par 017...
Aviso: la columna 'propensity_score' ya existe y será sobreescrita.
propensity_score() — n=1663 (T=1070, C=593)
  PS: min=0.4299  p25=0.6014  media=0.6434  p75=0.6792  max=0.8565
balance() — 4 filas sin PS descartadas.

──────────────────────────────────────────────────
  balance() | metodo='matching' | n=1663 (T=1070, C=593)
──────────────────────────────────────────────────
Matching 1:1 — 593 pares  (tratados=593, controles=593)
──────────────────────────────────────────────────

Procesando par 018...
Aviso: la columna 'propensity_score' ya existe y s

,n_tratados,n_controles,naive,regresion,regresion_ic95,regresion_pvalor,g_formula,ht,hajek,msm,msm_ic95,msm_pvalor,dr,pregunta_sq
0,18,18,0.4444,0.4253,"(0.1302, 0.7204)",0.0062,0.4117,-25.4593,0.3957,0.3957,"(-1.1868, 1.9783)",0.6146,-1.1347,001
1,121,121,0.3388,0.3409,"(0.2221, 0.4596)",0.0000,0.3409,-2.6971,0.3316,0.3316,"(0.091, 0.5723)",0.0071,0.2941,002
2,122,122,0.2131,0.2229,"(0.0998, 0.3461)",0.0004,0.2227,-1.9112,0.2016,0.2016,"(-0.0253, 0.4286)",0.0814,0.1378,003
3,81,81,0.5309,0.5608,"(0.4267, 0.6949)",0.0000,0.5571,-3.0073,0.6025,0.6025,"(0.2622, 0.9428)",0.0006,1.1003,004
4,796,796,0.4309,0.4051,"(0.3631, 0.4471)",0.0000,0.4054,0.3755,0.4072,0.4072,"(0.366, 0.4485)",0.0000,0.4073,005
5,306,306,0.4739,0.4734,"(0.4059, 0.5408)",0.0000,0.4734,1.5167,0.4381,0.4381,"(0.3435, 0.5327)",0.0000,0.4241,006
6,38,38,0.1316,0.0508,"(-0.1554, 0.257)",0.6247,0.0509,-16.6728,0.0531,0.0531,"(-0.577, 0.6832)",0.8671,-0.6184,007
7,52,52,0.1923,0.2350,"(0.0971, 0.3728)",0.0010,0.2351,-11.3281,0.2335,0.2335,"(-0.2664, 0.7333)",0.3564,0.4561,008
8,21,21,0.2381,0.2813,"(-0.0746, 0.6373)",0.1178,0.2892,-13.7667,0.3453,0.3453,"(-1.0142, 1.7049)",0.6105,3.1032,010
9,731,731,0.3516,0.3514,"(0.3115, 0.3912)",0.0000,0.3512,0.5336,0.3498,0.3498,"(0.3107, 0.3888)",0.0000,0.3511,012


# Lalonde

In [9]:
df = pd.read_excel(r"C:\Users\afpue\OneDrive\Documentos\GitHub\causalidad\Codigo organizado\csp.xlsx")

In [10]:
df.head()

,treatment,age,education,black,hispanic,married,nodegree,re74,re75,re78,u74,u75
0,1,37,11,1,0,1,1,0.0,0.0,9930.0460,1,1
1,1,22,9,0,1,0,1,0.0,0.0,3595.8940,1,1
2,1,30,12,1,0,0,0,0.0,0.0,24909.4500,1,1
3,1,27,11,1,0,0,1,0.0,0.0,7506.1460,1,1
4,1,33,8,1,0,0,1,0.0,0.0,289.7899,1,1


In [11]:
tratamiento = 'treatment'
#tratamiento = 'treat'
resultado   = 're78'
covariables = ['age', 'education', 'black', 'hispanic', 'married', 'nodegree', 're74', 're75', 'u74', 'u75']
#covariables = ['age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75', 'u74', 'u75']
#covariables = ['age']

In [12]:
# 1. Estimar PS
df = cs.propensity_score(df, tratamiento=tratamiento, covariables=covariables)

propensity_score() — n=614 (T=185, C=429)
  PS: min=0.0082  p25=0.0371  media=0.3013  p75=0.5810  max=0.9092
  Aviso: 212 unidades (34.5%) con PS < 0.05 o PS > 0.95 — considera trimming o truncating.


In [13]:
# 2. Balancear
df_trim  = cs.balance(df, tratamiento, metodo='trimming')
df_trunc = cs.balance(df, tratamiento, metodo='truncating')
df_match = cs.balance(df, tratamiento, metodo='matching')
df_sub   = cs.balance(df, tratamiento, metodo='subclassif')


──────────────────────────────────────────────────
  balance() | metodo='trimming' | n=614 (T=185, C=429)
──────────────────────────────────────────────────
Trimming (kappa=0.05) — 212 unidades eliminadas (34.5%)  -> n final=402  (tratados=182, controles=220)
  Rango PS conservado: [0.0501, 0.9092]
──────────────────────────────────────────────────


──────────────────────────────────────────────────
  balance() | metodo='truncating' | n=614 (T=185, C=429)
──────────────────────────────────────────────────
Truncating (kappa=0.05) — 212 PS truncados (34.5%)  -> n=614 (sin eliminar filas)
  PS original:  [0.0082, 0.9092]  ->  PS truncado: [0.0500, 0.9092]
──────────────────────────────────────────────────


──────────────────────────────────────────────────
  balance() | metodo='matching' | n=614 (T=185, C=429)
──────────────────────────────────────────────────
Matching 1:1 — 185 pares  (tratados=185, controles=185)
──────────────────────────────────────────────────


──────────────────

In [14]:
# 3. Calcular ATE
for nombre, datos in [('Original',         df),
                      ('Trimming',         df_trim),
                      ('Truncating',       df_trunc),
                      ('Matching',         df_match),
                      ('Subclasificación', df_sub)]:

    res = cs.calcular_ate(datos, resultado, tratamiento, covariables)
    if res is not None:
        print(f"\n── {nombre} ──────────────────────────")
        print(f"  n  tratados : {res['n_tratados']}")
        print(f"  n  controles: {res['n_controles']}")
        print(f"  Naive       : {res['naive']:.2f}")
        print(f"  Regresión   : {res['regresion']:.2f}  IC95={res['regresion_ic95']}  p={res['regresion_pvalor']:.4f}")
        print(f"  G-fórmula   : {res['g_formula']:.2f}")
        print(f"  HT          : {res['ht']:.2f}")
        print(f"  Hajek       : {res['hajek']:.2f}")
        print(f"  MSM         : {res['msm']:.2f}  IC95={res['msm_ic95']}  p={res['msm_pvalor']:.4f}")
        print(f"  DR          : {res['dr']:.2f}")



── Original ──────────────────────────
  n  tratados : 185
  n  controles: 429
  Naive       : -635.03
  Regresión   : 865.59  IC95=(-705.5648, 2436.7361)  p=0.2797
  G-fórmula   : 865.59
  HT          : -1200.68
  Hajek       : -412.92
  MSM         : -412.92  IC95=(-1642.0987, 816.2551)  p=0.5097
  DR          : -20.11

── Trimming ──────────────────────────
  n  tratados : 182
  n  controles: 220
  Naive       : 127.05
  Regresión   : 867.16  IC95=(-817.4065, 2551.721)  p=0.3121
  G-fórmula   : 867.16
  HT          : 1065.75
  Hajek       : 730.82
  MSM         : 730.82  IC95=(-607.1084, 2068.7492)  p=0.2835
  DR          : 674.52

── Truncating ──────────────────────────
  n  tratados : 185
  n  controles: 429
  Naive       : -635.03
  Regresión   : 865.59  IC95=(-705.5648, 2436.7361)  p=0.2797
  G-fórmula   : 865.59
  HT          : -1537.81
  Hajek       : -74.21
  MSM         : -74.21  IC95=(-1342.2921, 1193.8701)  p=0.9085
  DR          : 489.01

── Matching ───────────────────

In [15]:
import pandas as pd
import causalidad as cs

# ── Variables originales ──────────────────────────────────────
tratamiento = 'treat'
resultado   = 're78'
covariables = ['age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75', 'u74', 'u75']

# ── DataFrame auxiliar con transformaciones del slide 86 ──────
# I(education^2), I(education^3)  → columnas nuevas
# factor(black/hispanic/nodegree) → dtype category

df_lalonde = df.copy()
df_lalonde['educ2']  = df_lalonde['educ'] ** 2
df_lalonde['educ3']  = df_lalonde['educ'] ** 3
df_lalonde['black']  = df_lalonde['black'].astype('category')
df_lalonde['hisp']   = df_lalonde['hisp'].astype('category')
df_lalonde['nodegr'] = df_lalonde['nodegr'].astype('category')

# Covariables del modelo rico (igual al slide)
covariables_lalonde = ['age', 'educ', 'educ2', 'educ3',
                       'nodegr', 'black', 'hisp',
                       're74', 're75']

# ── PS sobre el df auxiliar ───────────────────────────────────
df_lalonde = cs.propensity_score(df_lalonde, tratamiento=tratamiento,
                                  covariables=covariables_lalonde)

# ── Balanceo ──────────────────────────────────────────────────
df_trim  = cs.balance(df_lalonde, tratamiento, metodo='trimming')
df_trunc = cs.balance(df_lalonde, tratamiento, metodo='truncating')
df_match = cs.balance(df_lalonde, tratamiento, metodo='matching')
df_sub   = cs.balance(df_lalonde, tratamiento, metodo='subclassif')

# ── ATE ───────────────────────────────────────────────────────
for nombre, datos in [('Original',         df_lalonde),
                      ('Trimming',         df_trim),
                      ('Truncating',       df_trunc),
                      ('Matching',         df_match),
                      ('Subclasificación', df_sub)]:

    res = cs.calcular_ate(datos, resultado, tratamiento, covariables_lalonde)
    if res is not None:
        print(f"\n── {nombre} ──────────────────────────")
        print(f"  n  tratados : {res['n_tratados']}")
        print(f"  n  controles: {res['n_controles']}")
        print(f"  Naive       : {res['naive']:.2f}")
        print(f"  Regresión   : {res['regresion']:.2f}  IC95={res['regresion_ic95']}  p={res['regresion_pvalor']:.4f}")
        print(f"  G-fórmula   : {res['g_formula']:.2f}")
        print(f"  HT          : {res['ht']:.2f}")
        print(f"  Hajek       : {res['hajek']:.2f}")
        print(f"  MSM         : {res['msm']:.2f}  IC95={res['msm_ic95']}  p={res['msm_pvalor']:.4f}")
        print(f"  DR          : {res['dr']:.2f}")

KeyError: 'educ'